In [0]:
"""
Module: Silver Configuration
Description: Sets up catalog variables and handles Spark optimization settings 
             for joining large datasets.
"""
dbutils.widgets.text("catalog_name", "nyc_taxi_dev", "Target Catalog")
CATALOG = dbutils.widgets.get("catalog_name")

# Optimization: Enable Adaptive Query Execution for large joins
# spark.conf.set("spark.sql.adaptive.enabled", "true")

print(f"🚀 Silver Layer initialized for catalog: {CATALOG}")

In [0]:
"""
Module: Task C1 - Incremental Weather Transformation
Description: Casts raw strings to proper types and merges into Silver.
Target: {CATALOG}.silver.dim_weather
"""
from delta.tables import DeltaTable
from pyspark.sql.functions import col

target_weather = f"{CATALOG}.silver.dim_weather"
df_bronze_weather = spark.read.table(f"{CATALOG}.bronze.raw_weather")

# 1. Transform and Type Casting
df_weather_typed = df_bronze_weather.select(
    col("date").cast("date"),
    col("rain_mm").cast("double").alias("precipitation_mm"),
    col("tempmax").cast("double").alias("temp_max"),
    col("tempmin").cast("double").alias("temp_min"),
    col("avg_temp").cast("double")
).fillna(0, subset=["precipitation_mm"])

# 2. Incremental Merge
if not spark.catalog.tableExists(target_weather):
    df_weather_typed.write.format("delta").saveAsTable(target_weather)
else:
    dt = DeltaTable.forName(spark, target_weather)
    dt.alias("t").merge(
        df_weather_typed.alias("s"), "t.date = s.date"
    ).whenNotMatchedInsertAll().execute()

print("✅ Weather Dimensions updated incrementally.")

In [0]:
"""
Module: Task C2 - Incremental Taxi Transformation & Quality Gates
Description: 1. Filters for only 'New' data in Bronze using ingested_at.
             2. Applies Quality Gates (Distance > 0, Fares > 0).
             3. Upserts into Silver using Delta MERGE.
Target: {CATALOG}.silver.fact_taxi_trips
"""
from delta.tables import DeltaTable
from pyspark.sql.functions import col

target_fact = f"{CATALOG}.silver.fact_taxi_trips"
bronze_raw = f"{CATALOG}.bronze.raw_trips"

# 1. Determine the Incremental High Watermark
if not spark.catalog.tableExists(target_fact):
    last_ingestion = "1900-01-01 00:00:00"
else:
    last_ingestion = spark.sql(f"SELECT MAX(ingested_at) FROM {target_fact}").collect()[0][0]

# 2. Fetch only new records and apply Data Quality Gates
df_new_trips = spark.read.table(bronze_raw) \
    .filter(col("ingested_at") > last_ingestion) \
    .select(
        col("trip_hash_id"),
        col("VendorID").cast("int").alias("vendor_id"),
        col("tpep_pickup_datetime").cast("timestamp").alias("pickup_time"),
        col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_time"),
        col("passenger_count").cast("int"),
        col("trip_distance").cast("double"),
        col("PULocationID").cast("int").alias("pickup_zip"),
        col("DOLocationID").cast("int").alias("dropoff_zip"),
        col("fare_amount").cast("double"),
        col("total_amount").cast("double"),
        col("ingested_at")
    ) \
    .filter(
        (col("trip_distance") > 0) & 
        (col("total_amount") > 0) & 
        (col("pickup_time") < col("dropoff_time"))
    )

# 3. Incremental MERGE
if not spark.catalog.tableExists(target_fact):
    df_new_trips.write.format("delta").saveAsTable(target_fact)
    print("🚀 Initial Silver Fact table created.")
else:
    dt_fact = DeltaTable.forName(spark, target_fact)
    dt_fact.alias("t").merge(
        df_new_trips.alias("s"), "t.trip_hash_id = s.trip_hash_id"
    ).whenNotMatchedInsertAll().execute()
    print(f"🔄 Merged {df_new_trips.count()} new cleaned records into Silver.")

In [0]:
"""
Module: Task C3 - Silver Enrichment Join
Description: Joins cleaned Taxi history with Weather history on Pickup Date.
Target: {CATALOG}.silver.enriched_taxi_weather
"""
from pyspark.sql.functions import to_date

fact = spark.read.table(f"{CATALOG}.silver.fact_taxi_trips")
weather = spark.read.table(f"{CATALOG}.silver.dim_weather")

# Left join to preserve taxi data even if weather is missing for a specific date
df_enriched = fact.join(
    weather, 
    to_date(fact.pickup_time) == weather.date, 
    "left"
).drop(weather.date)

df_enriched.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.enriched_taxi_weather")

print("🏆 Silver Layer Enriched and Ready for Gold Aggregates.")